# Lab 6.5: Upward and Downward Continuation — Stability and Applications

| | |
|---|---|
| **Module** | M6.5 — Upward and Downward Continuation Methods |
| **Estimated time** | ~2.5 hours |
| **Prerequisites** | Lab 6 (Parts 1–2); Homework 6.5, Problems 6.5.1–6.5.2 |
| **Textbook** | Blakely Ch. 12 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Derive** why upward continuation is stable and downward continuation is not, from the filter spectrum
2. **Implement** stable downward continuation with a low-pass regularization cutoff and justify the choice of cutoff
3. **Apply** upward continuation to separate overlapping regional and residual anomalies from multi-depth sources
4. **Quantify** noise amplification as a function of continuation distance and wavenumber

---

## How to use this notebook

- **`[PROVIDED]`** — run as-is
- **`[IMPLEMENT]`** — replace `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check; prints ✓ PASS or ✗ FAIL
- **`[EXPLORE]`** — starting point; modify freely

Hints: `print(hints['key'])`.


## Background

### The continuation filters

From Laplace's equation in the Fourier domain, the potential at height $z = z_0 + h$
is related to the potential at $z = z_0$ by (Blakely §12.1):

$$
\tilde{F}(k, h) = \tilde{F}(k, 0) \cdot e^{-2\pi k h}
$$

For **upward** continuation ($h > 0$): the factor $e^{-2\pi kh}$ is always
$\leq 1$, decaying to zero as $k \to \infty$.  This is a stable low-pass filter.

For **downward** continuation ($h < 0$, i.e., $-h > 0$): the factor becomes
$e^{+2\pi k|h|}$, which **grows exponentially** with $k$.  Any noise present
at high wavenumbers is amplified by this factor — potentially catastrophically.

### Stabilized downward continuation

To make downward continuation tractable, we apply a **regularized inverse filter**.
One common approach: apply the exact filter $e^{+2\pi kh}$ only for wavenumbers
below a cutoff $k_c$, and zero it above:

$$
W_{dc}(k) = \begin{cases}
  e^{+2\pi k|h|} & k \leq k_c \\
  0              & k > k_c
\end{cases}
$$

Alternatively, use a **Wiener-type** stabilizer:
$W_{dc} = e^{+2\pi k|h|} / (1 + \varepsilon^2 e^{4\pi k|h|})$,
where $\varepsilon$ is a small noise-to-signal parameter.

The choice of $k_c$ (or $\varepsilon$) involves a fundamental trade-off:
- Too small a $k_c$: the downward-continued field loses resolution (over-smoothed)
- Too large a $k_c$: noise is amplified and dominates the result

### Regional-residual separation

Upward continuation is an effective tool for **regional-residual separation**:
continuing up to a height large compared to shallow-source depths attenuates the
shallow sources while preserving the deep (regional) signal.  The residual
(original minus upward-continued) then highlights shallow sources.


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})

def _check(label, got, expected, rtol=1e-4, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  ✓ PASS  {label}')
    else:
        print(f'  ✗ FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
hints = {
    'p1_filter_plot': (
        "Plot exp(-2*pi*k*h) for h = 500, 1000, 2000 m (upward)\n"
        "and exp(+2*pi*k*h) for the same h (downward, uncutoff) on the same axes.\n"
        "k ranges from 0 to k_Nyquist = 1/(2*dx).\n"
        "Use a log scale on y to make the exponential growth visible."
    ),
    'p2_cutoff_choice': (
        "The amplification factor at cutoff k_c is exp(2*pi*k_c*h).\n"
        "For the result to be useful, you want this factor to be at most\n"
        "the signal-to-noise ratio (SNR).\n"
        "If SNR = 100 and h = 1 km:\n"
        "  k_c = ln(SNR) / (2*pi*h) = ln(100) / (2*pi*1000) ≈ 7.3e-4 cycles/m\n"
        "i.e., k_c ≈ 1 cycle per 1370 m."
    ),
    'p2_wiener': (
        "Wiener stabilizer:\n"
        "  W(k) = exp(+2*pi*k*h) / (1 + eps^2 * exp(4*pi*k*h))\n"
        "where eps is the noise-to-signal amplitude ratio (e.g., 0.01 for SNR=100).\n"
        "As k -> 0: W -> exp(2*pi*k*h) * 1 (exact downward continuation)\n"
        "As k -> inf: W -> 1/eps^2 * exp(-2*pi*k*h) (rolls off)"
    ),
    'p3_regional': (
        "Strategy: upward continue the total field to height h_reg >> depth_shallow.\n"
        "The upward-continued field approximates the regional (deep source only).\n"
        "The residual = original - upward_continued is dominated by shallow sources.\n"
        "Choose h_reg empirically: too small = shallow source bleeds into regional;\n"
        "too large = deep source is also attenuated."
    ),
}
# print(hints['p2_cutoff_choice'])

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Synthetic 1D gravity profile with two overlapping sources:
#   - Shallow source: sphere at depth 1.5 km, radius 500 m, drho = 300 kg/m^3
#   - Deep source (regional): sphere at depth 10 km, radius 4000 m, drho = 100 kg/m^3

G_grav = 6.674e-11
dx1d   = 200.0   # m, 1D profile spacing
x1d    = np.linspace(-20e3, 20e3, 201)   # m

def _sphere_gz_1d(x, depth, R, drho):
    M = (4/3)*np.pi*R**3*drho
    return G_grav * M * depth / (x**2 + depth**2)**1.5

rng = np.random.default_rng(99)
g_shallow  = _sphere_gz_1d(x1d, 1500.0, 500.0,  300.0) * 1e5   # mGal
g_deep     = _sphere_gz_1d(x1d, 10000.0, 4000.0, 100.0) * 1e5  # mGal
noise_sigma = 0.02   # mGal
g_total    = g_shallow + g_deep + rng.normal(0, noise_sigma, size=x1d.shape)

print(f'Shallow source peak: {g_shallow.max()*1e5:.2f} μGal  ({g_shallow.max():.4f} mGal)')
print(f'Deep source peak:    {g_deep.max():.4f} mGal')
print(f'Total range:         {g_total.min():.4f} to {g_total.max():.4f} mGal')

# Wavenumber array for 1D
k1d = np.fft.fftfreq(len(x1d), d=dx1d)   # cycles/m
K1d = np.abs(k1d)

---
## Part 1: Stability Analysis of the Continuation Filters *(Guided — ~35 min)*

Before implementing downward continuation, we need to understand *why* it
is unstable and what limits the continuation depth.

**Available hint:** `p1_filter_plot`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Compute upward and downward continuation filter amplitudes.

def continuation_filter(k, h):
    """
    Continuation filter amplitude in the wavenumber domain.

    Parameters
    ----------
    k : float or ndarray
        Radial wavenumber, cycles/m.  Must be >= 0.
    h : float
        Continuation height, meters.
        Positive = upward (stable, < 1).
        Negative = downward (unstable, > 1 for |k| > 0).

    Returns
    -------
    W : float or ndarray
        Filter amplitude.  W = exp(-2*pi*k*h).

    Notes
    -----
    - For h > 0: W < 1 for all k > 0 (stable low-pass)
    - For h < 0: W > 1 for all k > 0 (unstable amplification)
    - At k = 0 (DC): W = 1 always
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# At k=0: filter = 1 regardless of h
_check('W(k=0, h=1000) = 1', continuation_filter(0.0, 1000.0), 1.0)
_check('W(k=0, h=-500) = 1', continuation_filter(0.0, -500.0), 1.0)

# Upward h=500: at k=1/1000 (wavelength 1 km), W = exp(-pi) ≈ 0.0432
_k_test = 1.0/1000.0   # cycles/m
_W_up = continuation_filter(_k_test, 500.0)
_check('Upward W at k=1/1000, h=500', _W_up, np.exp(-2*np.pi*_k_test*500.0), rtol=1e-10)

# Downward: W_down = 1/W_up for same |h|
_W_down = continuation_filter(_k_test, -500.0)
_check('Downward W = 1/upward W', _W_down, 1.0/_W_up, rtol=1e-10)

# Upward W should be < 1 for k > 0, h > 0
_k_arr = np.linspace(1e-5, 2e-3, 100)
if np.all(continuation_filter(_k_arr, 1000.0) < 1.0):
    print('  ✓ PASS  Upward filter < 1 for all k > 0')
else:
    print('  ✗ FAIL  Upward filter should be < 1 for k > 0')

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Plot the continuation filter amplitudes to visualize stability.
#
# Make two panels:
#
# Panel 1 (linear scale): Upward continuation filters W(k) for h = 500, 1000, 2000 m.
#   x-axis: wavenumber in cycles/km,  y-axis: filter amplitude (linear)
#   Shade the area k > k_Nyquist in gray.
#
# Panel 2 (log y-scale): Downward continuation filter exp(+2*pi*k*h) for same |h|.
#   Also show the SNR-based cutoff for SNR=100 and h=1000 m:
#     k_c = ln(100) / (2*pi*1000) [cycles/m]
#   Mark k_c with a vertical dashed line.

k_plot = np.linspace(0, 1/(2*dx1d), 500)   # cycles/m up to Nyquist

# YOUR CODE HERE

### Question 1.1 — Quantifying instability

From your filter plot: at $h = 1000$ m and $k = k_{Nyquist}$ (corresponding to
the shortest resolvable wavelength at $\Delta x = 200$ m), what is the
amplification factor of the downward filter?

If the data noise has amplitude $\sigma = 0.02$ mGal and the signal amplitude
is $\sim 1$ mGal, at what wavenumber $k_c$ does the downward-continued noise
equal the signal amplitude?  Show your calculation.

**Your response:**

> *(Write 4–5 sentences, including the calculation.)*


---
## Part 2: Stabilized Downward Continuation *(Supported — ~50 min)*

We implement two regularization strategies for downward continuation and
compare them on noisy synthetic data.

**Available hints:** `p2_cutoff_choice`, `p2_wiener`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Stabilized downward continuation (1D).

def downward_continuation_cutoff(data, dx, h_down, k_cutoff):
    """
    Downward continue a 1D field using a wavenumber cutoff.

    Parameters
    ----------
    data : ndarray, shape (N,)
        Field values along profile.
    dx : float
        Spatial sampling interval, meters.
    h_down : float
        Downward distance (positive = downward = deeper), meters.
    k_cutoff : float
        Wavenumber cutoff, cycles/m.  Filter = 0 for |k| > k_cutoff.

    Returns
    -------
    data_down : ndarray
        Downward-continued field.

    Notes
    -----
    Filter: W(k) = exp(+2*pi*|k|*h_down) for |k| <= k_cutoff, else 0.
    h_down > 0 means moving the observation level downward (toward sources).
    """
    raise NotImplementedError


def downward_continuation_wiener(data, dx, h_down, eps):
    """
    Downward continue using a Wiener-type stabilizer.

    Parameters
    ----------
    data : ndarray, shape (N,)
    dx : float
        Sampling interval, meters.
    h_down : float
        Downward distance, meters (positive = deeper).
    eps : float
        Noise-to-signal amplitude ratio (e.g., 0.01 for SNR=100).

    Returns
    -------
    data_down : ndarray

    Notes
    -----
    W(k) = exp(+2*pi*|k|*h_down) / (1 + eps^2 * exp(4*pi*|k|*h_down))
    At k=0: W=1.  As k->inf: W->0 (stabilized).
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# Upward-then-downward should approximately recover the original (within the cutoff bandwidth)
from numpy.fft import fft, ifft, fftfreq

def _upward_1d(data, dx, h):
    k = fftfreq(len(data), d=dx)
    W = np.exp(-2*np.pi*np.abs(k)*h)
    return ifft(fft(data)*W).real

_h_test   = 1000.0
_k_cutoff = np.log(50.0) / (2*np.pi*_h_test)   # SNR=50

# Apply upward continuation then downward continuation
_g_up   = _upward_1d(g_total, dx1d, _h_test)
_g_down_co = downward_continuation_cutoff(_g_up, dx1d, _h_test, _k_cutoff)
_g_down_wi = downward_continuation_wiener(_g_up, dx1d, _h_test, eps=0.02)

# The low-wavenumber content should be recovered (check variance)
_var_orig  = np.var(g_total)
_var_round = np.var(_g_down_co)
if 0.1*_var_orig < _var_round < 2.0*_var_orig:
    print(f'  ✓ PASS  Round-trip variance reasonable: {_var_round:.4f} vs {_var_orig:.4f}')
else:
    print(f'  ✗ FAIL  Round-trip variance out of range: {_var_round:.4f} vs {_var_orig:.4f}')

# Wiener filter at k=0: should equal the DC value
_dc_orig = g_total.mean()
_dc_wi   = _g_down_wi.mean()
_check('Wiener: DC preserved', _dc_wi, _dc_orig, rtol=0.01)

# Downward continuation without cutoff should blow up
_g_no_cutoff = downward_continuation_cutoff(g_total, dx1d, 5000.0, k_cutoff=1/(2*dx1d))
if np.max(np.abs(_g_no_cutoff)) > 1e6:
    print('  ✓ PASS  Uncutoff downward continuation explodes (as expected)')
else:
    print('  ~ NOTE  Uncutoff continuation did not explode — check if you applied k_cutoff=Nyquist')

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Noise amplification study: how does downward continuation degrade with depth?
#
# 1. Start with g_shallow (the clean, noise-free shallow signal).
# 2. Add different noise levels: sigma = 0.01, 0.05, 0.1, 0.5 mGal.
# 3. For each noise level, apply downward continuation to h_down = 1500 m
#    (trying to re-image the source at depth 1.5 km from above).
#    Use the SNR-based cutoff: k_c = ln(SNR) / (2*pi*h_down)
#    where SNR = peak(g_shallow) / sigma.
# 4. Compare the downward-continued result to the upward-continued version
#    of the EXACT field at depth 1500 m (which you know from the forward model).
# 5. Plot RMS error vs. noise sigma.
#
# Requirements:
#   - Show 4 profiles (one per noise level) in a single figure
#   - Show the true signal at depth as a dashed black reference

# True signal at depth 1500 m: field from a point source at depth 1500 m,
# observed from a surface 1500 m above (i.e., the original surface)
# ... is just g_shallow. But we want the field *at* 1500 m depth from
# observing 1500 m further down. Use upward continuation in reverse:

# For the study, the 'truth' at depth h_down is the downward continued
# version of the *clean* signal:
h_down_target = 1500.0   # m
_k_c_clean = np.log(1e6) / (2*np.pi*h_down_target)   # very high SNR = no cutoff
g_true_at_depth = downward_continuation_cutoff(g_shallow, dx1d, h_down_target, _k_c_clean)

# YOUR CODE HERE

### Question 2.1 — Cutoff choice strategy

You used $k_c = \ln(\text{SNR}) / (2\pi h)$ to set the cutoff.
Explain the physical meaning of this formula.  At what wavenumber does
the amplification factor equal the SNR?  Why is this a natural choice
for the cutoff?  What is the main drawback of this approach?

**Your response:**

> *(Write 4–6 sentences.)*


### Question 2.2 — Wiener vs. cutoff

Compare the cutoff-based and Wiener-based downward continuation results.
Which produces fewer artifacts?  Is there a difference in how they handle
wavenumbers near the cutoff?  Why might the Wiener stabilizer be preferred
in practice despite being slightly more complex to implement?

**Your response:**

> *(Write 3–5 sentences.)*


---
## Part 3: Regional-Residual Separation *(Open — ~40 min)*

The synthetic profile `g_total` contains two overlapping anomalies:
a narrow shallow one ($d = 1.5$ km) and a broad deep one ($d = 10$ km).
In real surveys, these would represent a local structure of interest
(e.g., a salt dome) superimposed on a regional gradient (e.g., crustal
thickness variation).

Your goal: separate the two signals using upward continuation, and assess
how well you recovered each one.

**Available hint:** `p3_regional`


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Regional-residual separation.
#
# Strategy:
#   1. Choose an upward continuation height h_reg between the two source depths.
#      The deep source (10 km) should survive; the shallow source (1.5 km) should not.
#   2. g_regional = upward_continuation_1d(g_total, dx1d, h_reg)
#   3. g_residual = g_total - g_regional
#
# Implement upward_continuation_1d (same as the 2D version but in 1D).
# Try at least three values of h_reg: 3, 5, 8 km.
#
# Make a 3-row figure for each h_reg value:
#   Row 1: g_total, g_regional, and the true g_deep on same axes
#   Row 2: g_residual and the true g_shallow on same axes
#   Row 3: residuals (error) for both regional and residual estimates
#
# Compute RMS error for each separation.

def upward_continuation_1d(data, dx, h):
    """
    Upward continue a 1D profile by height h.
    """
    raise NotImplementedError

# YOUR CODE HERE

### Question 3.1 — Optimal continuation height

Which value of $h_{reg}$ gave the best separation?  How does the RMS error
in the regional estimate compare to the RMS error in the residual estimate?
Is there a systematic bias in how upward continuation estimates the regional,
and what causes it?

**Your response:**

> *(Write 4–6 sentences. Include RMS error numbers from your calculations.)*


### Question 3.2 — Limits of the method

The upward-continuation approach to regional-residual separation has a
fundamental limitation: it cannot separate two sources at the same depth.
Give a concrete example of a geological scenario where you would face this
problem, and describe an alternative strategy that might work in that case.

**Your response:**

> *(Write 4–5 sentences.)*


---
## Synthesis

### S1 — Connecting continuation to Laplace's equation

Upward continuation is a consequence of Laplace's equation having smooth
solutions (no sources above the measurement surface).  Explain **in physical
terms** why this means information about the sources can only flow upward
(from measurements to predictions), not downward without additional constraints.
What mathematical property of Laplace's equation guarantees this?

**Your response:**

> *(Write 4–6 sentences. Reference Green's functions or the maximum principle if you know them.)*

### S2 — Practical recommendations

A colleague asks you: "I have ground-based gravity data with station spacing
100 m and noise $\sigma = 0.05$ mGal.  I want to downward continue to 500 m
depth to better image a shallow target.  Is this feasible?"

Using the cutoff-wavenumber formula, estimate the spatial resolution
(minimum resolvable wavelength) of the downward-continued result.
Is this useful resolution for a target at 500 m depth?  What would you
recommend?

**Your response:**

> *(Write 4–6 sentences. Include a calculation.)*

### S3 — Information content and survey design

Both downward continuation and inversion (Lab 5) attempt to recover
subsurface structure from surface measurements.  Compare these two approaches:
what does each assume about the source geometry, what type of prior information
does each require, and when would you choose one over the other?

**Your response:**

> *(Write 5–7 sentences.)*


---
## Extensions *(optional)*

### E1 — Iterative downward continuation

An alternative to the direct inverse filter is iterative upward continuation:
start with the surface data as the initial guess for the depth-level field,
then iteratively correct using the difference between the upward-continued
trial field and the observed data.  Implement 5–10 iterations and compare
the result to the direct Wiener filter.

### E2 — Equivalent source method

The equivalent source method places a layer of point sources just below the
surface, fits them to the observed data, then predicts the field at any
height.  Implement a simplified version (1D, $N$ sources at depth 100 m)
and use it to interpolate the gravity profile to twice the station density.
Compare to simple cubic interpolation.

### E3 — 2D regional-residual separation

Apply the regional-residual separation to the 2D grid from Lab 6 (`F_grid`).
The three sources are at depths 1.5, 2.5, and 4.0 km.  Find a continuation
height that separates the deepest source from the two shallower ones,
and map both the regional and residual anomalies.


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Extension workspace.

# YOUR CODE HERE